# NDT7 (M-Lab) Data Prep — Malaysia Broadband + Mobile, State x Quarter

Aggregates `../../../data/ndt7/my/mlab_my_clean.parquet` (6,637,691 rows) into state x quarter
format, split into Broadband and Mobile/Cellular, using the same zoom-16 tile-binning +
weighted-aggregation SQL as the other small-country NDT7 prep notebooks (KH/LA/MM).

**Reliability:** `is_reliable = total_tests >= 100` — ไม่ใช้ `n_tiles` เหมือนทุกประเทศ NDT7
(Ookla ยังใช้ `n_tiles >= 5` อยู่ อย่าเอาไปเทียบกันตรง ๆ) · คอลัมน์ `n_tiles` ยังคำนวณและเก็บไว้

**State name mapping** — raw parquet เขียนชื่อบางรัฐติดกันหรือสะกดคนละแบบกับ
`malaysia_reference.csv` (`PulauPinang`→`Penang`, `Melaka`→`Malacca`, `Trengganu`→`Terengganu`,
`KualaLumpur`, `NegeriSembilan`) — `PROVINCE_MAP` ด้านล่างครอบคลุมครบ ตรวจกับ parquet จริงแล้ว

> **หมายเหตุ v2:** มาเลเซียเป็นประเทศที่สัดส่วน cellular เปลี่ยนจาก v1 มากที่สุด
> **46.3% → 31.9% (−14.4 pp)** ตกจากอันดับ 1 ไปอันดับ 4 ของภูมิภาค สาเหตุหลักคือ Starlink (96%)
> และ Yes/YTL (69%) ถูก ip-api ย้ายไปเป็น fixed · CelcomDigi มีสองสาย (Celcom cellular 63% /
> Digi broadband 60%) ต้องมี footnote ถ้าจะรวมเป็นเจ้าเดียว

**Outputs:**
- `data/exports/ndt7_malaysia_province_quarterly.csv` — Broadband
- `data/exports/ndt7_mobile_malaysia_province_quarterly.csv` — Mobile/Cellular


In [1]:
import duckdb
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PARQUET = '../../../data/ndt7/my/mlab_my_clean.parquet'
MY_REF_CSV = '../../../data/reference/malaysia_reference.csv'

ZOOM = 16
N_TILES = 2 ** ZOOM
MIN_TILE_TESTS = 3

### 1. Tile-Binning + Province-Quarter Aggregation (DuckDB)

All heavy row-level work (filtering, quarter-labeling, zoom-16 mercator tile assignment, GROUP BY tile x quarter x type x network_type) happens in one DuckDB SQL query against the raw parquet — no Python-side batching.

In [2]:
con = duckdb.connect()
con.execute("SET memory_limit='10GB'")                        # PH/ID ใหญ่ ต้องตั้ง
con.execute("SET temp_directory='../../../.tmp/duckdb'")   # ที่พักตอน spill
con.execute("SET preserve_insertion_order=false")

sql = f"""
WITH filtered AS (
    SELECT
        mean_throughput_mbps,
        min_rtt,
        type, network_type, province,
        year,
        CAST(CEIL(month / 3.0) AS INT) AS qtr,
        -- zoom-16 Web Mercator tile — ใช้เป็นคอลัมน์วินิจฉัยเท่านั้น ไม่ได้ใช้คิดค่าเฉลี่ย
        CAST(FLOOR((longitude + 180) / 360 * 65536) AS BIGINT) AS tx,
        CAST(FLOOR((1 - (ln(tan(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878)))
             + 1.0/cos(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878))))) / pi()) / 2 * 65536) AS BIGINT) AS ty
    FROM read_parquet('{RAW_PARQUET}')
    WHERE mean_throughput_mbps > 0
      AND province IS NOT NULL
      AND network_type IN ('broadband', 'cellular')
)
SELECT
    province, network_type, type,
    (CAST(year AS VARCHAR) || '-Q' || CAST(qtr AS VARCHAR)) AS year_q,
    AVG(mean_throughput_mbps)                       AS avg_thr,
    AVG(CASE WHEN min_rtt < 2000 THEN min_rtt END)  AS avg_lat,
    COUNT(*)                                        AS test_count,
    COUNT(DISTINCT CAST(tx AS VARCHAR) || '_' || CAST(ty AS VARCHAR)) AS n_tiles
FROM filtered
GROUP BY province, network_type, type, year_q
"""

tile_agg_all = con.execute(sql).df()
print(f"province x quarter x type x network rows: {len(tile_agg_all):,}")
print(f"quarters: {len(tile_agg_all['year_q'].unique())} | province: {tile_agg_all['province'].nunique()}")
print(tile_agg_all.groupby('network_type')['test_count'].sum().apply(lambda x: f'{x:,}'))

province x quarter x type x network rows: 610
quarters: 12 | province: 16
network_type
broadband    4,193,107
cellular     2,115,967
Name: test_count, dtype: object


### 2. Province Name Mapping — Raw (Lao romanization) → Reference (`province_en`)

Applied here, after DuckDB's tile-level aggregation — the intermediate result is small (thousands of rows, not hundreds of thousands), so this stays a plain pandas `.map()` exactly like the original.

In [3]:
# Raw NDT7 parquet 'province' values -> malaysia_reference.csv 'province_en'.
# มี 5 ชื่อที่เขียนต่างกัน (ติดกัน/สะกดคนละแบบ) ที่เหลือ 11 ชื่อตรงอยู่แล้ว
# ตรวจกับ parquet จริงแล้ว ครบทั้ง 16 รัฐ ไม่มีตกหล่น
PROVINCE_MAP = {
    'KualaLumpur': 'Kuala Lumpur',
    'NegeriSembilan': 'Negeri Sembilan',
    'PulauPinang': 'Penang',
    'Melaka': 'Malacca',
    'Trengganu': 'Terengganu',
}

tile_agg_all['province'] = tile_agg_all['province'].replace(PROVINCE_MAP)
_ref_names = set(pd.read_csv(MY_REF_CSV)['province_en'])
_unmapped = set(tile_agg_all['province']) - _ref_names
assert not _unmapped, f"มีชื่อรัฐที่ยังไม่ถูก map: {_unmapped}"
print(f"Applied PROVINCE_MAP ({len(PROVINCE_MAP)} entries) | "
      f"{tile_agg_all['province'].nunique()} รัฐ ตรงกับ reference ครบ")

Applied PROVINCE_MAP (5 entries) | 16 รัฐ ตรงกับ reference ครบ


### 3. Province-Level Weighted Aggregation (per network type)

In [4]:
def build_province_quarterly(tile_agg_all, network_type, ref):
    d = tile_agg_all[tile_agg_all['network_type'] == network_type]
    print(f"[{network_type}] province x quarter x type rows: {len(d):,}")

    dl = d[d['type'] == 'download'].rename(columns={
        'avg_thr': 'avg_d_mbps', 'avg_lat': 'avg_lat_ms_wt', 'test_count': 'total_tests'})
    ul = d[d['type'] == 'upload'].rename(columns={'avg_thr': 'avg_u_mbps'})

    dl_stats = dl[['year_q', 'province', 'avg_d_mbps', 'avg_lat_ms_wt', 'total_tests', 'n_tiles']]
    ul_stats = ul[['year_q', 'province', 'avg_u_mbps']]

    master = pd.merge(dl_stats, ul_stats, on=['year_q', 'province'], how='outer')
    master = master.rename(columns={'year_q': 'quarter'})
    master['year'] = master['quarter'].str.slice(0, 4).astype(int)
    master['quarter.1'] = master['quarter'].str.slice(6, 7).astype(int)

    # NDT7 ใช้ total_tests อย่างเดียว ไม่ใช้ n_tiles เป็นเกณฑ์ (Ookla ยังใช้ทั้งคู่)
    # เหตุผล: NDT7 ได้พิกัดจาก MaxMind ซึ่งเป็น city centroid ทุก test ในเมืองเดียวกันจึงตกลง tile
    # เดียวกัน n_tiles จึงวัด "จังหวัดนี้มีกี่เมืองใน MaxMind" ไม่ได้วัดการกระจายตัวของข้อมูล
    # (ลาวทั้งประเทศมีพิกัดต่างกัน 33 จุด n_tiles สูงสุด = 3 -> เกณฑ์ >=5 เป็นไปไม่ได้)
    # คอลัมน์ n_tiles ยังเก็บไว้ให้ดูใน "Data Quality" ของ EDA
    master['is_reliable'] = master['total_tests'] >= 100
    print(f"[{network_type}] province x quarter rows: {len(master)} | "
          f"reliable: {master['is_reliable'].sum()} ({master['is_reliable'].mean():.1%})")

    master = master.merge(
        ref[['province_en', 'region', 'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021',
             'density_per_km2', 'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']],
        left_on='province', right_on='province_en', how='left'
    ).drop(columns=['province_en'])

    missing_ref = master[master['region'].isna()]['province'].unique()
    if len(missing_ref):
        print(f"[{network_type}] WARNING — no reference match: {list(missing_ref)}")

    return master


EXPORT_COLS = ['province', 'quarter', 'year', 'quarter.1', 'avg_d_mbps', 'avg_u_mbps',
               'avg_lat_ms_wt', 'total_tests', 'n_tiles', 'is_reliable', 'region',
               'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021', 'density_per_km2',
               'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']

In [5]:
ref = pd.read_csv(MY_REF_CSV)

---
## Part 1 — Broadband

In [6]:
broadband_master = build_province_quarterly(tile_agg_all, 'broadband', ref)
broadband_master.head()

[broadband] province x quarter x type rows: 380
[broadband] province x quarter rows: 190 | reliable: 165 (86.8%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Johor,77.615875,56.280980,1859,41,30.984977,2023,1,True,Southern,3,4186300,8810,218,24097.67,771125.44
1,2023-Q1,Kedah,65.029587,63.297312,353,28,29.367652,2023,1,True,Northern,4,2217500,5694,233,15574.59,498386.88
2,2023-Q1,Kelantan,25.750683,62.158830,1203,16,10.083132,2023,1,True,East Coast,4,1888500,3764,125,10295.53,329456.96
3,2023-Q1,Kuala Lumpur,88.840522,53.796679,14209,46,31.391203,2023,1,True,Central,1,2067500,26882,8508,73529.35,2352939.20
4,2023-Q1,Labuan,20.273833,135.088180,61,2,7.896503,2023,1,False,Borneo / East Malaysia,1,100800,19648,1108,53742.45,1719758.40


In [7]:
out_bb = broadband_master[EXPORT_COLS].copy()
OUT_PATH_BB = '../../../data/exports/ndt7_malaysia_province_quarterly.csv'
out_bb.to_csv(OUT_PATH_BB, index=False)
print(f"Exported {len(out_bb)} rows -> {OUT_PATH_BB}")
out_bb.head(3)

Exported 190 rows -> ../../../data/exports/ndt7_malaysia_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Johor,2023-Q1,2023,1,77.615875,30.984977,56.280980,1859,41,True,Southern,3,4186300,8810,218,24097.67,771125.44
1,Kedah,2023-Q1,2023,1,65.029587,29.367652,63.297312,353,28,True,Northern,4,2217500,5694,233,15574.59,498386.88
2,Kelantan,2023-Q1,2023,1,25.750683,10.083132,62.158830,1203,16,True,East Coast,4,1888500,3764,125,10295.53,329456.96


---
## Part 2 — Mobile/Cellular

In [8]:
mobile_master = build_province_quarterly(tile_agg_all, 'cellular', ref)
mobile_master.head()

[cellular] province x quarter x type rows: 230
[cellular] province x quarter rows: 116 | reliable: 69 (59.5%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Johor,10.837890,91.031247,73.0,7.0,5.368582,2023,1,False,Southern,3,4186300,8810,218,24097.67,771125.44
1,2023-Q1,Kedah,31.371260,147.879933,15.0,5.0,5.978059,2023,1,False,Northern,4,2217500,5694,233,15574.59,498386.88
2,2023-Q1,Kelantan,16.003902,88.673000,7.0,4.0,9.125669,2023,1,False,East Coast,4,1888500,3764,125,10295.53,329456.96
3,2023-Q1,Kuala Lumpur,15.802485,111.278438,6260.0,17.0,5.852093,2023,1,True,Central,1,2067500,26882,8508,73529.35,2352939.20
4,2023-Q1,Malacca,16.079679,147.980826,23.0,5.0,6.323482,2023,1,False,Southern,2,1047100,10775,629,29472.46,943118.72


In [9]:
out_mb = mobile_master[EXPORT_COLS].copy()
OUT_PATH_MB = '../../../data/exports/ndt7_mobile_malaysia_province_quarterly.csv'
out_mb.to_csv(OUT_PATH_MB, index=False)
print(f"Exported {len(out_mb)} rows -> {OUT_PATH_MB}")
out_mb.head(3)

Exported 116 rows -> ../../../data/exports/ndt7_mobile_malaysia_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Johor,2023-Q1,2023,1,10.837890,5.368582,91.031247,73.0,7.0,False,Southern,3,4186300,8810,218,24097.67,771125.44
1,Kedah,2023-Q1,2023,1,31.371260,5.978059,147.879933,15.0,5.0,False,Northern,4,2217500,5694,233,15574.59,498386.88
2,Kelantan,2023-Q1,2023,1,16.003902,9.125669,88.673000,7.0,4.0,False,East Coast,4,1888500,3764,125,10295.53,329456.96


## Summary

- Input: Malaysia NDT7 raw test records, already province-joined + ISP-classified
- Output: province x quarter aggregates for Broadband and Mobile separately, tile-binned at
  Ookla's zoom-16 resolution, same `is_reliable` threshold as every Ookla country notebook and
  the other NDT7 "tigger" prep notebooks
- Engine: DuckDB (was: manual pyarrow-batch-streaming loop in pandas) — verified to reproduce
  the prior pandas-based export exactly (float-precision-only differences)